# Results Dashboard (Plotly)

This notebook visualizes training/evaluation/benchmark outputs with Plotly.

Expected artifacts (generated by `benchmark.py`):
- `benchmark_runs/baseline/benchmark_report.json`
- `benchmark_runs/candidate/benchmark_report.json`


In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path('..').resolve()
BASELINE = ROOT / 'benchmark_runs' / 'baseline' / 'benchmark_report.json'
CANDIDATE = ROOT / 'benchmark_runs' / 'candidate' / 'benchmark_report.json'

def load_report(path):
    if not path.exists():
        print(f'Missing: {path}')
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

baseline = load_report(BASELINE)
candidate = load_report(CANDIDATE)
baseline is not None, candidate is not None


In [ ]:
def aggregate_means(report):
    if not report:
        return {}
    return report.get('aggregate', {}).get('means', {})

b = aggregate_means(baseline)
c = aggregate_means(candidate)
metrics = [
    'train_time_per_epoch',
    'time_to_best_val_loss',
    'inference_latency_ms',
    'throughput_images_per_sec',
    'mean_iou',
    'binary_iou',
    'counting_bias',
    'count_agreement_spread',
    'model_size_mb',
]
rows = []
for m in metrics:
    rows.append({'metric': m, 'baseline': b.get(m), 'candidate': c.get(m)})
df_compare = pd.DataFrame(rows)
df_compare


In [ ]:
if len(df_compare):
    fig = go.Figure()
    fig.add_bar(name='Baseline', x=df_compare['metric'], y=df_compare['baseline'])
    fig.add_bar(name='Candidate', x=df_compare['metric'], y=df_compare['candidate'])
    fig.update_layout(title='Baseline vs Candidate Metric Comparison', barmode='group', xaxis_tickangle=-45, height=600)
    fig.show()


In [ ]:
def trials_df(report, label):
    if not report:
        return pd.DataFrame()
    t = report.get('trials', [])
    d = pd.DataFrame(t)
    if len(d):
        d['run'] = label
    return d

df_trials = pd.concat([trials_df(baseline, 'baseline'), trials_df(candidate, 'candidate')], ignore_index=True)
df_trials[['run','trial_id','train_time_per_epoch','inference_latency_ms','mean_iou','throughput_images_per_sec']] if len(df_trials) else df_trials


In [ ]:
if len(df_trials):
    fig = px.scatter(
        df_trials,
        x='inference_latency_ms',
        y='mean_iou',
        color='run',
        size='throughput_images_per_sec',
        hover_data=['trial_id','train_time_per_epoch'],
        title='Accuracy-Speed Pareto View (Lower latency, higher IoU)'
    )
    fig.show()


In [ ]:
def bottleneck_df(report, run_name):
    if not report:
        return pd.DataFrame()
    rows = report.get('aggregate', {}).get('prioritized_bottlenecks', [])
    out = pd.DataFrame(rows)
    if len(out):
        out['run'] = run_name
    return out

df_b = pd.concat([bottleneck_df(baseline, 'baseline'), bottleneck_df(candidate, 'candidate')], ignore_index=True)
if len(df_b):
    fig = px.bar(df_b, x='stage', y='avg_seconds', color='run', barmode='group', title='Prioritized Bottlenecks')
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print('No bottleneck data found yet.')
